<div class='alert alert-warning'>

SciPy's interactive examples with Jupyterlite are experimental and may not always work as expected. Execution of cells containing imports may result in large downloads (up to 60MB of content for the first import from SciPy). Load times when importing from SciPy may take roughly 10-20 seconds. If you notice any problems, feel free to open an [issue](https://github.com/scipy/scipy/issues/new/choose).

</div>

Taking the derivative of the antiderivative recovers the original
spline, up to floating point error:


In [ ]:
import numpy as np
from scipy.interpolate import make_interp_spline
x = np.linspace(0, np.pi/2, 70)
y = 1 / np.sqrt(1 - 0.8*np.sin(x)**2)
spl = make_interp_spline(x, y)    # spl is a cubic spline
spl(1.7), spl.antiderivative().derivative()(1.7)

(array(2.15654299), array(2.15654299))

Antiderivative can be used to evaluate definite integrals:


In [ ]:
ispl = spl.antiderivative()
ispl(np.pi/2) - ispl(0)

np.float64(2.25720536)

This is indeed an approximation to the complete elliptic integral
$K(m) = \int_0^{\pi/2} [1 - m\sin^2 x]^{-1/2} dx$:


In [ ]:
from scipy.special import ellipk
ellipk(0.8)

np.float64(2.25720533)

The coefficient array of the returned spline is padded with ``k + 1``
extra trailing entries, which are not used in evaluations, as the
examples below demonstrate:


In [ ]:
from scipy.interpolate import BSpline, make_interp_spline
x = np.arange(11)
y = x**4 + np.sin(x)
spl = make_interp_spline(x, y, k=3)  # spl is a cubic (degree 3) spline

A spline of degree ``k`` is built from polynomial pieces of degree
``k``, joined at the knots ``t``. The number of knots, the number of
coefficients, and the degree are related by
``len(t) - len(c) == k + 1``. Padded coefficients instead satisfy
``len(c) == len(t)``:


In [ ]:
len(spl.t), len(spl.c), spl.k    # knots, coefficients, degree of spl

(15, 11, 3)

In [ ]:
a = spl.antiderivative()         # a is a quartic spline
len(a.t), len(a.c), a.k          # knots, coefficients, degree of a

(17, 17, 4)

In [ ]:
a.c[len(a.t) - a.k - 1:]         # repeat padding, unused in evaluations

array([20002.23910041, 20002.23910041, 20002.23910041, 20002.23910041,
       20002.23910041])

Slice off the padding to get exactly ``len(a.t) - a.k - 1`` coefficients:


In [ ]:
c = a.c[:len(a.t) - a.k - 1]
len(c)

12

The trimmed coefficients define the same spline, since the padding
never enters evaluations. Evaluating the spline before (``a``) and
after (``BSpline(a.t, c, a.k)``) trimming on a grid of points spanning
the data range gives the same values:


In [ ]:
xx = np.linspace(0, 10, 51)
np.allclose(BSpline(a.t, c, a.k)(xx), a(xx))

True